# Clase 3 — Implementación de RAG

## De qué se trata

RAG conecta recuperación con generación. No reemplaza al LLM: le entrega evidencia privada, actualizable y trazable antes de responder.

## 1. Dos pipelines que no se deben mezclar

| Pipeline de datos | Pipeline de consulta |
|---|---|
| Carga, limpieza, chunks, embeddings, persistencia | Pregunta, embedding, retrieval, contexto, LLM, validación |
| Corre al cambiar fuentes/modelo/chunking | Corre por cada interacción |
| Métricas: cobertura, chunks, errores | Métricas: recall, grounding, latencia |

In [ ]:
# Ejemplo ejecutable: Clase 3 — Implementación de RAG
import matplotlib.pyplot as plt

lanes = {
    "ingesta": ["fuente", "limpiar", "chunking", "embeddings", "índice"],
    "consulta": ["pregunta", "embedding", "Top-K", "contexto", "respuesta"],
}
fig, ax = plt.subplots(figsize=(10, 3))
for y, (name, steps) in enumerate(lanes.items()):
    for x, step in enumerate(steps):
        ax.text(x, y, step, ha="center", va="center", bbox={"boxstyle":"round","fc":"#dbe9f6"})
        if x: ax.annotate("", (x-.25,y), (x-.7,y), arrowprops={"arrowstyle":"->"})
    ax.text(-1.2, y, name, weight="bold")
ax.set(xlim=(-1.8, 4.5), ylim=(-.6, 1.6), xticks=[], yticks=[], title="Pipeline de datos e inferencia")
plt.show()

## 2. Retrieval no es generación

Top-K entrega candidatos. Un threshold puede descartar evidencia débil. Un reranker puede reordenar pocos candidatos con más precisión. El prompt debe limitar al LLM a responder con ese contexto.

| Decisión | Beneficio | Riesgo si se usa sola |
|---|---|---|
| Top-K | Limita cantidad | Siempre entrega resultados |
| Threshold | Permite abstenerse | Puede eliminar todo |
| Reranking | Mejora precisión | Agrega latencia |
| Caché | Reduce costo | Puede quedar desactualizada |

In [ ]:
# Ejemplo ejecutable: Clase 3 — Implementación de RAG
import matplotlib.pyplot as plt

scores = [.91, .78, .49, .24]
colors = ["#2a9d8f", "#2a9d8f", "#e9c46a", "#e76f51"]
plt.bar(["c1","c2","c3","c4"], scores, color=colors)
plt.axhline(.5, color="black", linestyle="--", label="threshold")
plt.ylim(0, 1)
plt.title("Top-K y criterio de suficiencia")
plt.legend()
plt.show()

## 3. Grounding y contrato público

La respuesta debe incluir exactamente:

| Clave | Contenido |
|---|---|
| user_question | Pregunta original validada |
| system_answer | Respuesta grounded o abstención |
| chunks_related | Evidencia recuperada con ID, fuente y score |

Una pregunta sin evidencia no es un fallo del sistema si contesta con claridad que el contexto no alcanza.

## Ejemplo ejecutable

El bloque siguiente ilustra una implementación mínima del concepto.


In [ ]:
# Ejemplo ejecutable: Clase 3 — Implementación de RAG
def build_context(results):
    return "\n\n".join(
        f"[{r['chunk_id']} | {r['source']} | score={r['score']:.2f}]\n{r['content']}"
        for r in results
    )

results = [
    {"chunk_id":"faq-003", "source":"faq.txt", "score":.91,
     "content":"El enlace de contraseña vence en treinta minutos."},
    {"chunk_id":"faq-008", "source":"faq.txt", "score":.72,
     "content":"Soporte verifica el correo laboral registrado."},
]
response = {
    "user_question": "¿Cómo recupero mi contraseña?",
    "system_answer": "Usá el enlace enviado al correo laboral.",
    "chunks_related": results,
}
assert set(response) == {"user_question", "system_answer", "chunks_related"}
print(build_context(results))
print(response)

## Consideraciones técnicas

- Copiar valores o parámetros sin relacionarlos con el corpus y las consultas.
- Confundir una demo que funciona con una medición de calidad.
- Omitir IDs, fuentes, configuración o validaciones.